In [ ]:
import os
from pathlib import Path
import numpy as np
import SimpleITK as sitk
import pydicom
import matplotlib.pyplot as plt
import pydicom_seg as dcmseg
from radiomics import featureextractor
import pandas as pd

In [ ]:
# testing stuff with paths

general_dir = Path(os.path.expanduser('~/Documents/NSCLC-Radiomics'))
path_records = []

for patient_dir in general_dir.iterdir():
    #if not patient_dir.is_dir():
    #    continue

    scan_id = patient_dir.name

    for study_dir in patient_dir.iterdir():
        #if not study_dir.is_dir():
        #    continue

        ct_series = None
        seg_series = None

        for series_dir in study_dir.iterdir():
            if not series_dir.is_dir():
                continue

            if 'Segmentation' in series_dir.name and any(series_dir.glob('*.dcm')):
                seg_series = series_dir
                continue

            if any(series_dir.glob('*.dcm')) and len(list(series_dir.glob('*.dcm'))) >= 10:
                ct_series = series_dir

        if ct_series is not None and seg_series is not None:
            path_records.append({
                'scan_id': scan_id,
                'path_ct': ct_series,
                'path_mask':seg_series
            })
        else:
            print(f"Skipping {patient_dir.name}/{study_dir.name}: ct_series={ct_series is not None}, seg_series={seg_series is not None}")

path_df = pd.DataFrame(path_records, columns=['scan_id', 'path_ct', 'path_mask'])

In [ ]:
#input_dcm_new = sorted(input_dcm.glob('*.dcm')) #makes list of all the paths in the directory and sorts
#seg_dcm_new = sorted(seg_dcm.glob('*.dcm')) #makes list of all the paths in the directory and sorts

ser_reader = sitk.ImageSeriesReader()
i = 0
for ct_path, mask_path in zip(path_df['path_ct'], path_df['path_mask']):
    seg_file = sorted(mask_path.glob('*.dcm'))
    if not seg_file:
        print(f"No DICOM files found in mask folder: {mask_path}")
        continue

    seg = pydicom.dcmread(seg_file[0])
    reader = dcmseg.SegmentReader()
    result = reader.read(seg)
    seg_infos = result.segment_infos

    neoplasm_count = sum(
        1 for info in seg_infos.values()
        if 'Neoplasm' in info.get('SegmentLabel', '')
    )

    if neoplasm_count == 0:
        print(f"NEOPLASM NOT IN ANY LABELS for mask: {mask_path}")
        i += 1
    elif neoplasm_count > 1:
        print(f"NEOPLASM OCCURS {neoplasm_count} TIMES in labels for mask: {mask_path}")

print(i)
        #print(seg_num, info['SegmentLabel'])
    #segment_sequence = result.available_segments
    #print(segment_sequence)

    #for segment in segment_sequence:
    #    print(seg.SegmentSequence[segment - 1].SegmentLabel)